# 00 - From Supervised to Unsupervised

Everything in module 02 had a `y`. You knew the right answer for every training row, and
"good" meant matching it.

**Unsupervised learning has no `y`.** You have features only, and you are looking for
structure that was already in the data. Which means:

| | Supervised | Unsupervised |
|---|---|---|
| Input | `X` and `y` | `X` only |
| Goal | predict `y` for new rows | find structure in `X` |
| "Correct" | measurable against `y` | **there is no correct** |
| Evaluation | accuracy, ROC-AUC, RMSE | silhouette, inertia, and human judgement |
| Typical use | fraud detection, pricing, diagnosis | customer segments, recommendations, compression, anomaly detection |

That fourth row is the one that catches people. You cannot compute an accuracy. You have
to argue that the structure you found is useful - and the metrics below only help you
argue, they do not settle it.

This notebook gives you the two tools you will use everywhere: **k-means** (grouping rows)
and **PCA** (compressing columns). The notebooks that follow go much deeper.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

mall = pd.read_csv("../data/Mall_Customers.csv")
print(mall.shape)
mall.head()

## 1. Look before you cluster

There is no target column here. Just customers.

In [ ]:
print(mall.dtypes)
print()
print(mall.describe().round(1))

In [ ]:
features = ["Annual_Income_(k$)", "Spending_Score"]
X = mall[features]

plt.figure(figsize=(7, 5))
plt.scatter(X[features[0]], X[features[1]], s=35, color="#4C78A8")
plt.xlabel(features[0]); plt.ylabel(features[1])
plt.title("Can you see groups here? K-means will try to.")
plt.grid(alpha=.3)
plt.show()

## 2. K-means in four lines

The algorithm:
1. Drop `k` centre points at random.
2. Assign each row to its nearest centre.
3. Move each centre to the mean of its assigned rows.
4. Repeat 2-3 until nothing moves.

Notebook 01 makes you implement exactly this from scratch. Here is the library version.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# K-means measures distance, so scaling is mandatory - same lesson as notebook 02 in module 02.
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(X[features[0]], X[features[1]], c=labels, cmap="viridis", s=40)
centres = StandardScaler().fit(X).inverse_transform(kmeans.cluster_centers_)
plt.scatter(centres[:, 0], centres[:, 1], marker="X", s=250, c="red", label="centroids")
plt.xlabel(features[0]); plt.ylabel(features[1])
plt.title("K-means with k=5")
plt.legend(); plt.grid(alpha=.3)
plt.show()

## 3. But how do you choose `k`?

This is the central question of clustering, and there is no clean answer. Two tools help.

**Elbow method** - plot inertia (total squared distance to the nearest centroid) against
`k`. Inertia always falls as `k` rises, so you look for the bend where it stops falling fast.

**Silhouette score** - for each point, how much closer is it to its own cluster than to
the next nearest? Ranges from -1 to 1. Higher is better, and unlike inertia it has a peak.

In [ ]:
from sklearn.metrics import silhouette_score

ks = range(2, 11)
inertias, silhouettes = [], []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(ks, inertias, "o-", color="#4C78A8")
axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia")
axes[0].set_title("Elbow method - look for the bend"); axes[0].grid(alpha=.3)

axes[1].plot(ks, silhouettes, "o-", color="#F58518")
best_k = ks[int(np.argmax(silhouettes))]
axes[1].axvline(best_k, ls="--", c="grey", label=f"best k = {best_k}")
axes[1].set_xlabel("k"); axes[1].set_ylabel("silhouette score")
axes[1].set_title("Silhouette - this one has a real peak")
axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout()
plt.show()

> **The honest position.** These metrics measure whether the clusters are geometrically
> tidy. They cannot tell you whether the clusters are *meaningful to the business*.
> A silhouette of 0.55 with five segments the marketing team recognises beats a silhouette
> of 0.71 with two segments nobody can act on. **Always profile your clusters.**

## 4. Profile the clusters - the step people skip

In [ ]:
mall["cluster"] = labels

profile = mall.groupby("cluster")[["Age"] + features].mean()
profile["count"] = mall["cluster"].value_counts().sort_index()
profile.round(1)

Now name them. Something like "young high spenders, low income" or "wealthy but cautious".
If you cannot write a plausible name for a cluster, it probably is not a real segment.

## 5. PCA - compressing columns

K-means groups **rows**. PCA compresses **columns**: it finds new axes that capture as
much variance as possible, so you can plot 13 dimensions in 2.

In [ ]:
from sklearn.decomposition import PCA

# Use the heart-disease features from module 02 - 13 columns, impossible to plot directly.
heart = pd.read_csv("../data/heart-disease.csv")
Xh = StandardScaler().fit_transform(heart.drop("target", axis=1))

pca = PCA()
pca.fit(Xh)

cumulative = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.bar(range(1, len(cumulative) + 1), pca.explained_variance_ratio_,
        color="#4C78A8", label="per component")
plt.plot(range(1, len(cumulative) + 1), cumulative, "o-", color="#E45756", label="cumulative")
plt.axhline(0.9, ls="--", c="grey", label="90% of variance")
plt.xlabel("principal component"); plt.ylabel("explained variance ratio")
plt.title("How many components do you actually need?")
plt.legend(); plt.grid(alpha=.3)
plt.show()

print(f"components needed for 90% of the variance: {np.argmax(cumulative >= 0.9) + 1} of {Xh.shape[1]}")

In [ ]:
# Project 13 dimensions down to 2 and colour by the (unused) target, just to see.
coords = PCA(n_components=2).fit_transform(Xh)

plt.figure(figsize=(7, 5))
for value, colour, name in [(0, "#4C78A8", "no disease"), (1, "#E45756", "disease")]:
    mask = heart["target"] == value
    plt.scatter(coords[mask, 0], coords[mask, 1], s=30, c=colour, label=name, alpha=.7)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("PCA never saw the labels - yet the classes partly separate")
plt.legend(); plt.grid(alpha=.3)
plt.show()

**That is the point of PCA.** It found the directions of greatest variance using the
features alone, and those directions happen to line up with the thing we care about.
That is why PCA is useful as a preprocessing step: fewer columns, less noise, faster models.

**Warning:** PC1 and PC2 are mixtures of all original features. You gain a plot and lose
interpretability. Never PCA a dataset you need to explain to a regulator without a plan.

---
## Exercises

1. Re-run k-means on the mall data **without scaling**. Do the clusters change? Relate
   your answer to module 02, notebook 02.
2. Add `Age` as a third clustering feature. Does the best `k` by silhouette change?
   Profile the new clusters and try to name them.
3. Run k-means with `n_init=1` and several different `random_state` values. Do you get the
   same clusters every time? What does `n_init=10` protect you against?
4. Fit `PCA(n_components=5)` on the heart data, then train a logistic regression on those
   5 components. Compare accuracy against using all 13 features. What did you trade?
5. **Stretch:** cluster the heart-disease patients ignoring `target`, then cross-tabulate
   your clusters against the real `target`. Did unsupervised learning rediscover the disease?

---
## Next

- **01** - implement k-means yourself, from scratch. Do not skip it.
- **02** - PCA, DBSCAN and hierarchical clustering on real country data.
- **03** - collaborative filtering: recommendations from a ratings matrix.
- **04** - clustering text documents with TF-IDF.
- **05** - EDA on churn data, the bridge back to a supervised problem.